In [75]:
import numpy as np
import random
from dataclasses import dataclass
from fractions import Fraction
from itertools import combinations
from typing import Sequence 
import gurobipy as gp
from gurobipy import GRB

In [ ]:
# license = grbgetkey 9e853db1-a31f-4ef7-8144-5bfb1f1f8e20

In [56]:
@dataclass(frozen=True)
class Instance:
    t: tuple            # thresholds t_1..t_n  (Fractions)
    pi: tuple           # priors, sum to 1     (Fractions)
    tau: Fraction       # true decision threshold
    c: Fraction         # cost rate
    X: tuple            # agent types          (Fractions)
    D: tuple            # type weights, sum to 1 (Fractions)
 
    @property
    def n(self) -> int:
        return len(self.t)
 
    def __post_init__(self):
        assert len(self.pi) == self.n and len(self.D) == len(self.X)
        assert sum(self.pi) == 1, "priors must sum to 1"
        assert sum(self.D) == 1, "type weights must sum to 1"
        assert all(p > 0 for p in self.pi)

In [57]:
def uniform_instance(t, tau, c, m: int) -> Instance:
    """Uniform priors and a uniform grid of m types on [0,1]."""
    n = len(t)
    X = tuple(Fraction(k, m - 1) for k in range(m))
    return Instance(
        t=tuple(Fraction(v) for v in t),
        pi=tuple(Fraction(1, n) for _ in range(n)),
        tau=Fraction(tau),
        c=Fraction(c),
        X=X,
        D=tuple(Fraction(1, m) for _ in range(m)),
    )

In [58]:
def action_set(inst: Instance, x: Fraction) -> tuple:
    """A(x) = {x} union {t_i : t_i > x}.  Upward moves only, sorted ascending.
 
    Downward moves are weakly dominated (they cost nothing under (a-x)_+ but
    can only lose reward), and excluding them makes 'lowest cost' coincide with
    'smallest action', so the tie-break rule is unambiguous.
    """
    return tuple(sorted({x} | {ti for ti in inst.t if ti > x}))

In [59]:
def cost(inst: Instance, x: Fraction, a: Fraction) -> Fraction:
    return inst.c * max(a - x, Fraction(0))
 
 
def h(inst: Instance, i: int, a: Fraction) -> int:
    return 1 if a >= inst.t[i] else 0
 
 
def f_true(inst: Instance, x: Fraction) -> int:
    return 1 if x >= inst.tau else 0
 
 
def U(inst: Instance, i: int, a: Fraction, x: Fraction,
      eps: Fraction = Fraction(0)) -> Fraction:
    """Unnormalised utility contribution of classifier i, with tie-break eps."""
    return inst.pi[i] * (h(inst, i, a) - (1 + eps) * cost(inst, x, a))
 
 
def L(inst: Instance, i: int, a: Fraction, x: Fraction) -> int:
    """Loss coefficient: 1 iff classifier i's verdict on report a is wrong."""
    return 1 if h(inst, i, a) != f_true(inst, x) else 0
 
 
def big_M(inst: Instance, x: Fraction, a: Fraction, ap: Fraction,
          eps: Fraction) -> Fraction:
    """Tightest valid big-M for the triple (x, a, a').
 
    The constraint contains a SUBSET sum selected by x_ij, so the bound must be
    the max over subsets, i.e. the sum of the POSITIVE parts -- not the range of
    the full sum over all i (which is smaller whenever terms cancel).
    """
    return sum(max(U(inst, i, ap, x, eps) - U(inst, i, a, x, eps), Fraction(0))
               for i in range(inst.n))

In [60]:
def min_utility_gap(inst: Instance) -> Fraction:
    """Smallest nonzero gap in unperturbed block utility, over ALL blocks.
 
    Enumerates subsets, so only for small n.  Any eps in (0, delta_min / C),
    with C the max cost difference, leaves every strict preference intact while
    breaking every exact tie toward the cheaper action.
    """
    best = None
    for r in range(1, inst.n + 1):
        for S in combinations(range(inst.n), r):
            for x in inst.X:
                A = action_set(inst, x)
                vals = [sum(U(inst, i, a, x) for i in S) for a in A]
                for p in range(len(vals)):
                    for q in range(p + 1, len(vals)):
                        d = abs(vals[p] - vals[q])
                        if d > 0 and (best is None or d < best):
                            best = d
    return best if best is not None else Fraction(1)

In [61]:
def suggest_epsilon(inst: Instance) -> Fraction:
    """A certified eps: strictly inside (0, delta_min / C)."""
    delta = min_utility_gap(inst)
    C = max(cost(inst, x, a) for x in inst.X for a in action_set(inst, x))
    if C == 0:
        return Fraction(1, 2)
    return Fraction(1, 2) * delta / C

In [62]:

def set_partitions(elements: Sequence[int]):
    if len(elements) == 0:
        yield []
        return
    first, rest = elements[0], elements[1:]
    for smaller in set_partitions(rest):
        for k in range(len(smaller)):
            yield smaller[:k] + [[first] + smaller[k]] + smaller[k + 1:]
        yield [[first]] + smaller
 
 
def best_response(inst: Instance, block: Sequence[int], x: Fraction) -> Fraction:
    """Argmax of block utility over A(x); ties broken toward LOWER COST.
 
    A(x) is ascending and cost is strictly increasing in a, so scanning in order
    and keeping strict improvements returns the min-cost maximiser exactly.
    """
    A = action_set(inst, x)
    best_a, best_v = A[0], sum(U(inst, i, A[0], x) for i in block)
    for a in A[1:]:
        v = sum(U(inst, i, a, x) for i in block)
        if v > best_v:                     # strict: earlier (cheaper) a wins ties
            best_a, best_v = a, v
    return best_a
 
 
def partition_loss(inst: Instance, partition) -> Fraction:
    total = Fraction(0)
    for block in partition:
        for x, dx in zip(inst.X, inst.D):
            a = best_response(inst, block, x)
            for i in block:
                total += dx * inst.pi[i] * L(inst, i, a, x)
    return total
 
 
def brute_force_optimal(inst: Instance):
    best_p, best_v = None, None
    for p in set_partitions(list(range(inst.n))):
        v = partition_loss(inst, p)
        if best_v is None or v < best_v:
            best_p, best_v = p, v
    return best_v, best_p

In [63]:

def solve_milp(inst: Instance, eps: Fraction | None = None,
               linearization: str = "w", symmetry_breaking: bool = True,
               loose_M: Fraction | None = None, verbose: bool = False):
    """Build and solve the partition MILP.
 
    eps            tie-break perturbation; None -> certified value
    linearization  'w' (epigraph, O(mn) vars) or 'z' (literal product)
    loose_M        if given, use this constant M everywhere instead of the
                   tight per-triple value (for demonstrating the failure mode)
    """
    if eps is None:
        eps = suggest_epsilon(inst)
    n = inst.n
 
    env = gp.Env(empty=True)
    env.setParam("OutputFlag", 1 if verbose else 0)
    env.start()
    mdl = gp.Model("partition", env=env)
 
    # --- assignment ------------------------------------------------------
    xv = mdl.addVars(n, n, vtype=GRB.BINARY, name="x")
    mdl.addConstrs((gp.quicksum(xv[i, j] for j in range(n)) == 1
                    for i in range(n)), name="assign")
 
    if symmetry_breaking:
        # restricted-growth: classifier i may only open bucket <= i, and bucket
        # j requires bucket j-1 to be occupied by some earlier classifier.
        for i in range(n):
            for j in range(i + 1, n):
                mdl.addConstr(xv[i, j] == 0)
        for i in range(1, n):
            for j in range(1, i + 1):
                mdl.addConstr(xv[i, j] <= gp.quicksum(xv[ip, j - 1]
                                                      for ip in range(i)))
 
    # --- best response ---------------------------------------------------
    yv, acts = {}, {}
    for xi, x in enumerate(inst.X):
        A = action_set(inst, x)
        acts[xi] = A
        for j in range(n):
            for ai in range(len(A)):
                yv[xi, ai, j] = mdl.addVar(vtype=GRB.BINARY,
                                           name=f"y[{xi},{ai},{j}]")
            mdl.addConstr(gp.quicksum(yv[xi, ai, j]
                                      for ai in range(len(A))) == 1)
 
    for xi, x in enumerate(inst.X):
        A = acts[xi]
        for j in range(n):
            for ai, a in enumerate(A):
                ua = gp.quicksum(float(U(inst, i, a, x, eps)) * xv[i, j]
                                 for i in range(n))
                for api, ap in enumerate(A):
                    if api == ai:
                        continue
                    uap = gp.quicksum(float(U(inst, i, ap, x, eps)) * xv[i, j]
                                      for i in range(n))
                    M = (float(loose_M) if loose_M is not None
                         else float(big_M(inst, x, a, ap, eps)))
                    mdl.addConstr(ua >= uap - M * (1 - yv[xi, ai, j]))
 
    # --- objective -------------------------------------------------------
    if linearization == "w":
        wv = mdl.addVars(len(inst.X), n, lb=0.0, name="w")
        for xi, x in enumerate(inst.X):
            A = acts[xi]
            for j in range(n):
                for ai, a in enumerate(A):
                    lin = gp.quicksum(float(inst.pi[i] * L(inst, i, a, x))
                                      * xv[i, j] for i in range(n))
                    Mxa = float(sum(inst.pi[i] * L(inst, i, a, x)
                                    for i in range(n)))
                    mdl.addConstr(wv[xi, j] >= lin - Mxa * (1 - yv[xi, ai, j]))
        obj = gp.quicksum(float(inst.D[xi]) * wv[xi, j]
                          for xi in range(len(inst.X)) for j in range(n))
 
    elif linearization == "z":
        zv, terms = {}, []
        for xi, x in enumerate(inst.X):
            A = acts[xi]
            for j in range(n):
                for ai, a in enumerate(A):
                    for i in range(n):
                        coef = float(inst.D[xi] * inst.pi[i]) * L(inst, i, a, x)
                        if coef == 0.0:
                            continue          # only >=0 coefs appear; skip zeros
                        z = mdl.addVar(lb=0.0, ub=1.0, name=f"z[{i},{xi},{ai},{j}]")
                        zv[i, xi, ai, j] = z
                        # only the >= branch binds under minimisation
                        mdl.addConstr(z >= xv[i, j] + yv[xi, ai, j] - 1)
                        terms.append(coef * z)
        obj = gp.quicksum(terms)
    else:
        raise ValueError("linearization must be 'w' or 'z'")
 
    mdl.setObjective(obj, GRB.MINIMIZE)
    mdl.optimize()
 
    if mdl.Status != GRB.OPTIMAL:
        return None, None, mdl
 
    blocks = {}
    for i in range(n):
        for j in range(n):
            if xv[i, j].X > 0.5:
                blocks.setdefault(j, []).append(i)
    return mdl.ObjVal, [sorted(b) for b in blocks.values()], mdl

In [64]:
def check(inst, tag, lin="w"):
    bf_val, bf_part = brute_force_optimal(inst)
    ml_val, ml_part, _ = solve_milp(inst, linearization=lin)
    ok = abs(float(bf_val) - ml_val) < 1e-7
    print(f"  {tag:34s} brute={float(bf_val):.6f}  milp={ml_val:.6f}  "
          f"{'OK' if ok else 'MISMATCH'}   {ml_part}")
    return ok

In [74]:

print("=" * 78)
print("1. RANDOM INSTANCES  (n=3,4; both linearisations)")
print("=" * 78)
random.seed(7)
allok = True
for trial in range(8):
    # n = random.choice([3, 4])
    # m = random.choice([6, 9])
    n = 6
    m = 10
    t = sorted(random.sample([Fraction(k, 10) for k in range(1, 10)], n))
    tau = Fraction(random.randint(2, 8), 10)
    c = Fraction(random.randint(1, 6), 2)
    inst = uniform_instance(t, tau, c, m)
    lin = "w" if trial % 2 == 0 else "z"
    allok &= check(inst, f"n={n} m={m} c={c} tau={tau} [{lin}]", lin)
print(f"  --> all match: {allok}")

1. RANDOM INSTANCES  (n=3,4; both linearisations)
  n=6 m=10 c=5/2 tau=4/5 [w]         brute=0.383333  milp=0.383333  OK   [[0, 1, 2, 3, 4, 5]]
  n=6 m=10 c=1/2 tau=1/5 [z]         brute=0.183333  milp=0.183333  OK   [[0, 1, 5], [2, 3], [4]]
  n=6 m=10 c=5/2 tau=1/5 [w]         brute=0.150000  milp=0.150000  OK   [[0], [1, 2], [3], [4], [5]]
  n=6 m=10 c=5/2 tau=3/5 [z]         brute=0.233333  milp=0.233333  OK   [[0, 1, 2, 3, 4, 5]]
  n=6 m=10 c=2 tau=2/5 [w]           brute=0.133333  milp=0.133333  OK   [[0, 1, 2], [3, 4, 5]]
  n=6 m=10 c=5/2 tau=1/5 [z]         brute=0.166667  milp=0.166667  OK   [[0], [1, 2], [3], [4], [5]]
  n=6 m=10 c=1 tau=3/5 [w]           brute=0.500000  milp=0.500000  OK   [[0, 1, 2, 3, 5], [4]]


GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information

In [66]:
print("=" * 78)
print("2. THE TWO LINEARISATIONS AGREE  (same instance, 'w' vs 'z')")
print("=" * 78)
inst = uniform_instance([Fraction(2, 10), Fraction(5, 10), Fraction(8, 10)],
                        Fraction(4, 10), Fraction(2), 9)
vw, pw, mw = solve_milp(inst, linearization="w")
vz, pz, mz = solve_milp(inst, linearization="z")
print(f"  w-form: obj={vw:.6f}  vars={mw.NumVars:5d}  constrs={mw.NumConstrs:5d}")
print(f"  z-form: obj={vz:.6f}  vars={mz.NumVars:5d}  constrs={mz.NumConstrs:5d}")
print(f"  agree: {abs(vw - vz) < 1e-7}")

2. THE TWO LINEARISATIONS AGREE  (same instance, 'w' vs 'z')
  w-form: obj=0.222222  vars=  102  constrs=  228
  z-form: obj=0.222222  vars=  156  constrs=  243
  agree: True


In [69]:
print("=" * 78)
print("3. FAILURE MODE A -- optimistic tie-breaking (eps = 0)")
print("=" * 78)
# t=(.1,.2,.4), tau=.2, c=2.5: pooled block {1,2} leaves the qualified agent at
# x=.2 exactly indifferent between reporting .2 and .4.
inst = Instance(t=(Fraction(1, 10), Fraction(1, 5), Fraction(2, 5)),
                pi=tuple([Fraction(1, 3)] * 3),
                tau=Fraction(1, 5), c=Fraction(5, 2),
                X=tuple(Fraction(k, 10) for k in range(11)),
                D=tuple([Fraction(1, 11)] * 11))
x = Fraction(1, 5)
print(f"  block {{1,2}} (t=0.2,0.4), x={float(x)} (qualified):")
for a in action_set(inst, x):
    u = sum(U(inst, i, a, x) for i in (1, 2))
    err = sum(L(inst, i, a, x) for i in (1, 2))
    print(f"    a={float(a):.2f}  U_block={float(u):+.5f}  in-block errors={err}")
bf, bp = brute_force_optimal(inst)
v0, p0, _ = solve_milp(inst, eps=Fraction(0))
ve, pe, _ = solve_milp(inst)
print(f"  brute force (min-cost ties):     {float(bf):.6f}   {bp}")
print(f"  MILP with eps=0  (optimistic):   {v0:.6f}   {p0}")
print(f"  MILP with certified eps:         {ve:.6f}   {pe}")
print(f"  --> eps=0 understates OPT by {float(bf)-v0:.6f} (=1/33); "
      f"same partition returned, only the VALUE is wrong")

3. FAILURE MODE A -- optimistic tie-breaking (eps = 0)
  block {1,2} (t=0.2,0.4), x=0.2 (qualified):
    a=0.20  U_block=+0.33333  in-block errors=1
    a=0.40  U_block=+0.33333  in-block errors=0
  brute force (min-cost ties):     0.121212   [[0], [1, 2]]
  MILP with eps=0  (optimistic):   0.090909   [[0], [1, 2]]
  MILP with certified eps:         0.121212   [[0], [1, 2]]
  --> eps=0 understates OPT by 0.030303 (=1/33); same partition returned, only the VALUE is wrong


In [70]:
print("=" * 78)
print("4. FAILURE MODE B -- undersized big-M deletes valid partitions")
print("=" * 78)
inst = Instance(t=(Fraction(6, 10), Fraction(9, 10)),
                pi=(Fraction(1, 2), Fraction(1, 2)),
                tau=Fraction(1, 2), c=Fraction(3),
                X=(Fraction(1, 2), Fraction(1, 1)),
                D=(Fraction(1, 2), Fraction(1, 2)))
x = Fraction(1, 2)
a, ap = Fraction(9, 10), Fraction(6, 10)
d = [U(inst, i, ap, x) - U(inst, i, a, x) for i in range(2)]
print(f"  x={float(x)}, comparing a={float(a)} against a'={float(ap)}")
print(f"    per-classifier differences d_i = {[float(v) for v in d]}")
print(f"    full-sum range formula  : {float(sum(d)):.4f}  (cancels!)")
print(f"    correct positive-part   : {float(big_M(inst, x, a, ap, Fraction(0))):.4f}")
bf, bp = brute_force_optimal(inst)
vt, pt, _ = solve_milp(inst)
vl, pl, _ = solve_milp(inst, loose_M=Fraction(4, 10), symmetry_breaking=False)
print(f"  brute force:              {float(bf):.6f}   {bp}")
print(f"  MILP, tight per-triple M: {vt:.6f}   {pt}")
print(f"  MILP, M=0.4 (too small):  {vl:.6f}   {pl}" if vl is not None else "INFEASIBLE / partition deleted")

4. FAILURE MODE B -- undersized big-M deletes valid partitions
  x=0.5, comparing a=0.9 against a'=0.6
    per-classifier differences d_i = [0.45, -0.05]
    full-sum range formula  : 0.4000  (cancels!)
    correct positive-part   : 0.4500
  brute force:              0.250000   [[0, 1]]
  MILP, tight per-triple M: 0.250000   [[0, 1]]
INFEASIBLE / partition deleted


In [71]:
print("=" * 78)
print("5. MONOTONICITY + UP-SET STRUCTURE  (the beta_i claim)")
print("=" * 78)
inst = uniform_instance([Fraction(2, 10), Fraction(5, 10), Fraction(8, 10)],
                        Fraction(4, 10), Fraction(3, 2), 21)
bad = 0
for p in set_partitions(list(range(inst.n))):
    for block in p:
        zs = [best_response(inst, block, x) for x in inst.X]
        if any(zs[k + 1] < zs[k] for k in range(len(zs) - 1)):
            bad += 1
        for i in block:
            acc = [1 if zs[k] >= inst.t[i] else 0 for k in range(len(inst.X))]
            # up-set: once accepted, accepted forever
            if any(acc[k] > acc[k + 1] for k in range(len(acc) - 1)):
                bad += 1
print(f"  violations of monotone z* or up-set accept region: {bad}")

5. MONOTONICITY + UP-SET STRUCTURE  (the beta_i claim)
  violations of monotone z* or up-set accept region: 0


In [72]:
print("=" * 78)
print("6. LOSS DECOMPOSITION  L = sum_i pi_i * Pr_D[classifier i errs]")
print("=" * 78)
for p in list(set_partitions(list(range(inst.n))))[:4]:
    direct = partition_loss(inst, p)
    decomp = Fraction(0)
    for block in p:
        for i in block:
            err = sum(dx for x, dx in zip(inst.X, inst.D)
                      if L(inst, i, best_response(inst, block, x), x))
            decomp += inst.pi[i] * err
    print(f"  {str(p):28s} direct={float(direct):.6f}  decomp={float(decomp):.6f}"
          f"  {'OK' if direct == decomp else 'MISMATCH'}")

6. LOSS DECOMPOSITION  L = sum_i pi_i * Pr_D[classifier i errs]
  [[0, 1, 2]]                  direct=0.222222  decomp=0.222222  OK
  [[0], [1, 2]]                direct=0.285714  decomp=0.285714  OK
  [[0, 1], [2]]                direct=0.333333  decomp=0.333333  OK
  [[1], [0, 2]]                direct=0.285714  decomp=0.285714  OK


In [ ]:
print("=== basic usage, plain numpy floats ===")
thresholds = np.array([0.1, 0.2, 0.4])
priors     = np.array([1/3, 1/3, 1/3])
res = find_optimal_partition(thresholds, priors, 0.2, 2.5, np.linspace(0,1,11), check=True)
print(res)
print("  per-classifier error:", np.round(res.per_classifier_error,4))
print("  block mass Y_B      :", np.round(res.block_mass,4))
print("  epsilon used        :", f"{res.epsilon:.3e}")
print("  model size          :", res.n_vars, "vars,", res.n_constrs, "constrs")
 
print("\n=== unnormalised priors, unsorted thresholds ===")
r = find_optimal_partition(np.array([0.8,0.2,0.5]), np.array([2.,5.,3.]), 0.4, 1.5,
                           np.linspace(0,1,11), check=True)
print(r, " normalized priors:", np.round(r.extra["priors_normalized"],4))
print("  (indices refer to the ORIGINAL order: t[0]=0.8, t[1]=0.2, t[2]=0.5)")
 
print("\n=== MILP vs brute force across random instances ===")
rng = np.random.default_rng(0); ok = 0; tot = 0
for _ in range(12):
    n = rng.integers(3,5); m = int(rng.choice([11,21]))
    t = np.sort(rng.choice(np.arange(1,10)/10, size=n, replace=False))
    p = rng.integers(1,6,size=n).astype(float)
    tau = float(rng.choice(np.arange(2,9)/10)); c = float(rng.choice([0.5,1.,1.5,2.,2.5,3.]))
    Xg = np.linspace(0,1,m)
    a = find_optimal_partition(t,p,tau,c,Xg,method="milp")
    b = find_optimal_partition(t,p,tau,c,Xg,method="brute")
    tot += 1; ok += abs(a.loss-b.loss) < 1e-12
    if abs(a.loss-b.loss) >= 1e-12:
        print("  MISMATCH", t, p, tau, c, a.loss, b.loss)
print(f"  {ok}/{tot} agree exactly")
 
print("\n=== non-uniform weights ===")
Xg = np.linspace(0,1,11); w = np.exp(-3*Xg); 
r = find_optimal_partition(np.array([0.2,0.5,0.7]), np.ones(3), 0.45, 2.0, Xg,
                           weights=w, check=True)
print(r)
 
print("\n=== snapping warning fires on a genuinely irrational input ===")
with warnings.catch_warnings(record=True) as W:
    warnings.simplefilter("always")
    find_optimal_partition(np.array([np.pi/10, 0.5, 0.8]), np.ones(3), 0.4, 2.0,
                           np.linspace(0,1,6), max_denominator=3)
    print("  warnings:", [str(x.message)[:70] for x in W])
 